<a href="https://colab.research.google.com/github/ammar-aa/Fly_rank_internship_repo/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
"""
Section 1: Method choice and why

Method: Pairwise ranking via Logistic Regression on feature differences

There is no ground-truth label in this problem. The Week 4 baseline (score = -trend_pct * position_share) is itself a hand-written scoring rule built from two signals — trend_pct and gsc_sum_position — not a target to predict against. So this isn't classification or regression toward a known truth; it's a comparison between two different ways of scoring and ranking the same rows.

What actually matters for this problem is the order pages fall in, not the exact score value — the baseline's real job is to rank pages by refresh urgency. Pairwise ranking directly optimizes for "does page A outrank page B," which matches that goal more precisely than trying to hit an arbitrary numeric score.

To keep the comparison fair, the model uses the same two signals the baseline formula uses (trend_pct, gsc_sum_position) — no additional features, no leakage.

Logistic Regression is used because it's the simplest model that can learn a combination of the two signals. The baseline combines them multiplicatively with fixed, hand-picked weighting (-trend_pct * position_share). Training a Logistic Regression on pairwise feature differences tests whether a different, learned combination of the same two signals produces a meaningfully different — and possibly more sensible — ranking than the fixed formula.

Other menu methods don't fit as well here: clustering isn't appropriate since the goal isn't to discover groups, it's to compare two ranking systems; and Decision Tree / Random Forest / Gradient Boosting are heavier than needed for two features and would obscure the direct, interpretable weight comparison against the formula's fixed coefficients.
"""

'\nSection 1: Method choice and why\n\nMethod: Pairwise ranking via Logistic Regression on feature differences\n\nThere is no ground-truth label in this problem. The Week 4 baseline (score = -trend_pct * position_share) is itself a hand-written scoring rule built from two signals — trend_pct and gsc_sum_position — not a target to predict against. So this isn\'t classification or regression toward a known truth; it\'s a comparison between two different ways of scoring and ranking the same rows.\n\nWhat actually matters for this problem is the order pages fall in, not the exact score value — the baseline\'s real job is to rank pages by refresh urgency. Pairwise ranking directly optimizes for "does page A outrank page B," which matches that goal more precisely than trying to hit an arbitrary numeric score.\n\nTo keep the comparison fair, the model uses the same two signals the baseline formula uses (trend_pct, gsc_sum_position) — no additional features, no leakage.\n\nLogistic Regression 

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
"""
Section 2: Split design

Grouped by client, not time-aware.

Each content_hash_id appears exactly once in the dataset (verified: value_counts().max() == 1), so there is no repeated time series at the row level to be time-aware about. The Week 4 aggregation already collapsed the daily-grain source data into one summary row per page, with trend_pct and gsc_sum_position computed across the full time window per page. A time-aware split would be guarding against a leak that structurally cannot occur here, so it isn't used.

The real leak risk is client-level: multiple pages likely share the same client_hash_id, and if pages are split randomly, pages from the same client could land on both sides of train/test. Client-level effects (e.g. one client's whole site trending down for reasons unrelated to trend_pct or gsc_sum_position individually) could let the model partly learn "this client's pages behave a certain way" instead of the actual signal relationship being tested. Splitting by client_hash_id, so every page belonging to a given client stays entirely on one side, closes that leak.

Because this is a pairwise ranking setup, the split happens at the row level first, before pairs are generated: clients are divided into a train pool and a test pool, and only afterward are pairs sampled — separately — within each pool. This guarantees no single row, and no client, appears on both sides of any pair.
"""

'\nSection 2: Split design\n\nGrouped by client, not time-aware.\n\nEach content_hash_id appears exactly once in the dataset (verified: value_counts().max() == 1), so there is no repeated time series at the row level to be time-aware about. The Week 4 aggregation already collapsed the daily-grain source data into one summary row per page, with trend_pct and gsc_sum_position computed across the full time window per page. A time-aware split would be guarding against a leak that structurally cannot occur here, so it isn\'t used.\n\nThe real leak risk is client-level: multiple pages likely share the same client_hash_id, and if pages are split randomly, pages from the same client could land on both sides of train/test. Client-level effects (e.g. one client\'s whole site trending down for reasons unrelated to trend_pct or gsc_sum_position individually) could let the model partly learn "this client\'s pages behave a certain way" instead of the actual signal relationship being tested. Splittin

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
from google.colab import userdata
auth=userdata.get("HF_TOKEN")
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from scipy.stats import spearmanr
con=duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{auth}'
);
""")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

In [4]:
df = con.sql(f"""
SELECT *
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [5]:
dfF = con.sql(f"""
SELECT SUM(gsc_impressions) AS gsc_impressions, client_hash_id, content_hash_id
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet'
GROUP BY client_hash_id, content_hash_id
""").df()

dfM = con.sql(f"""
SELECT SUM(gsc_impressions) AS gsc_impressions, client_hash_id, content_hash_id
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
GROUP BY client_hash_id, content_hash_id
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [6]:
df_trend = dfM.merge(dfF, on=['client_hash_id', 'content_hash_id'], suffixes=('_feb', '_mar'), how='outer')

In [7]:
df_trend = df_trend[df_trend['gsc_impressions_feb'] >= 30]
df_trend = df_trend[df_trend['gsc_impressions_mar'] > 0]

df_trend['trend_pct'] = (
    (df_trend['gsc_impressions_mar'] - df_trend['gsc_impressions_feb'])
    / df_trend['gsc_impressions_feb']
) * 100

clip_value = df_trend['trend_pct'].quantile(0.99)
df_trend['trend_pct'] = df_trend['trend_pct'].clip(lower=-clip_value, upper=clip_value)

In [8]:
df = df.groupby(['client_hash_id', 'content_hash_id'], as_index=False).agg(
    gsc_sum_position=('gsc_sum_position', 'sum'),
    gsc_avg_position=('gsc_avg_position', 'mean'),
)

In [9]:
df = df.merge(df_trend[['client_hash_id', 'content_hash_id', 'trend_pct']], on=['client_hash_id', 'content_hash_id'], how='left')

In [10]:
negative_mean = df.loc[df['trend_pct'] < 0, 'trend_pct'].mean()
positive_mean = df.loc[df['trend_pct'] > 0, 'trend_pct'].mean()
conditions = [
    df['trend_pct'] < negative_mean,
    (df['trend_pct'] < 0) & (df['trend_pct'] >= negative_mean),
    (df['trend_pct'] >= 0) & (df['trend_pct'] < positive_mean),   # now includes 0
    df['trend_pct'] >= positive_mean
]
ranks = ['Sharp decline', 'Mild decline', 'Mild growth', 'Strong growth']
df['trend_dir'] = np.select(conditions, ranks, default=None)

In [11]:
position_share = df['gsc_sum_position'] / df['gsc_sum_position'].sum()

cap_value = position_share.quantile(0.99)
position_share_capped = position_share.clip(upper=cap_value)

score = -df['trend_pct'] * position_share_capped * 1000
df['score']=score
df['score'] = df['score'] * 1000

In [12]:
df = df[['content_hash_id', 'client_hash_id', 'trend_pct', 'gsc_sum_position', 'score']].copy()

In [13]:
conditions = [
    (df['trend_pct'] < negative_mean) & (position_share > position_share.median()),
    (df['trend_pct'] < negative_mean) & (position_share <= position_share.median()),
    (df['trend_pct'] >= negative_mean) & (df['trend_pct'] < 0) & (position_share > position_share.median()),
    (df['trend_pct'] >= negative_mean) & (df['trend_pct'] < 0),
    (df['trend_pct'] >= 0) & (position_share > position_share.median()),
]
codes = [
    'strong declining trend with low page position',
    'strong declining trend',
    'mild declining trend with low page position',
    'mild declining trend',
    'low page position',
]
df['reason_code'] = np.select(conditions, codes, default='STABLE')


In [14]:
df['action'] = np.select(
    [
        df['reason_code'] == 'strong declining trend with low page position',
        df['reason_code'].isin(['strong declining trend', 'mild declining trend with low page position']),
    ],
    ['REFRESH', 'MONITOR'],
    default='SKIP'
)

In [15]:
df = df.dropna(subset=['trend_pct', 'score']).reset_index(drop=True)
print("Rows after dropping NaN trend_pct/score:", len(df))

Rows after dropping NaN trend_pct/score: 105120


In [16]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_hash_id']))

df_train = df.iloc[train_idx].reset_index(drop=True)
df_test = df.iloc[test_idx].reset_index(drop=True)

print("Train rows:", len(df_train), "| Test rows:", len(df_test))
overlap = set(df_train['client_hash_id']) & set(df_test['client_hash_id'])
print("Client overlap (should be 0):", len(overlap))


Train rows: 99603 | Test rows: 5517
Client overlap (should be 0): 0


In [17]:
def make_balanced_pairs(data, n_pairs, seed, scaler=None, fit_scaler=False, cap_quantile=0.99):
    rng = np.random.default_rng(seed)
    idx_a = rng.integers(0, len(data), n_pairs)
    idx_b = rng.integers(0, len(data), n_pairs)

    mask = idx_a != idx_b
    idx_a, idx_b = idx_a[mask], idx_b[mask]

    trend = data['trend_pct'].values

    pos_raw = data['gsc_sum_position'].values
    cap_value = np.quantile(pos_raw, cap_quantile)
    pos_capped = np.clip(pos_raw, a_min=None, a_max=cap_value)

    interaction = trend * pos_capped

    feats = np.column_stack([trend, pos_capped, interaction])

    if fit_scaler:
        scaler = StandardScaler()
        feats = scaler.fit_transform(feats)
    else:
        feats = scaler.transform(feats)

    scores = data['score'].values
    label = (scores[idx_a] > scores[idx_b]).astype(int)

    swap = label == 0
    idx_a_final = np.where(swap, idx_b, idx_a)
    idx_b_final = np.where(swap, idx_a, idx_b)

    X_diff = feats[idx_a_final] - feats[idx_b_final]
    y = np.ones(len(idx_a_final), dtype=int)

    flip = rng.random(len(y)) < 0.5
    X_diff[flip] = -X_diff[flip]
    y[flip] = 0

    return X_diff, y, scaler

X_train, y_train, scaler = make_balanced_pairs(df_train, n_pairs=150_000, seed=42, fit_scaler=True)
X_test, y_test, _ = make_balanced_pairs(df_test, n_pairs=30_000, seed=99, scaler=scaler, fit_scaler=False)

model_int = LogisticRegression()
model_int.fit(X_train, y_train)
print("Test pair accuracy:", model_int.score(X_test, y_test))
print("Weights [trend, pos_capped, interaction]:", model_int.coef_[0])

Test pair accuracy: 0.9640285371382851
Weights [trend, pos_capped, interaction]: [ -0.73185832  -0.36648833 -51.73579221]


In [18]:
model = LogisticRegression()
model.fit(X_train, y_train)

train_acc_int = model.score(X_train, y_train)
test_acc_int = model.score(X_test, y_test)

print("Train pair accuracy:", train_acc_int)
print("Test pair accuracy:", test_acc_int)
print("Learned weights [trend_pct, gsc_sum_position, interaction]:", model.coef_[0])

Train pair accuracy: 0.979553060707476
Test pair accuracy: 0.9640285371382851
Learned weights [trend_pct, gsc_sum_position, interaction]: [ -0.73185832  -0.36648833 -51.73579221]


In [19]:
feats_raw = df[['trend_pct', 'gsc_sum_position']].values
interaction_raw = df['trend_pct'].values * df['gsc_sum_position'].values
feats_all = np.column_stack([feats_raw[:,0], feats_raw[:,1], interaction_raw])

feats_scaled = scaler.transform(feats_all)

w_int = model.coef_[0]
df['model_score_interaction'] = feats_scaled @ w_int

corr_int, pval_int = spearmanr(df['score'], df['model_score_interaction'])
print("Spearman correlation (baseline vs 3-feature model):", corr_int)

for k in [20, 50, 100, 500, 1000]:
    top_baseline = set(df.nlargest(k, 'score')['content_hash_id'])
    top_model_int = set(df.nlargest(k, 'model_score_interaction')['content_hash_id'])
    overlap = len(top_baseline & top_model_int) / k
    print(f"Top-{k} overlap: {overlap:.2%}")

Spearman correlation (baseline vs 3-feature model): 0.9960800300445092
Top-20 overlap: 5.00%
Top-50 overlap: 2.00%
Top-100 overlap: 11.00%
Top-500 overlap: 35.60%
Top-1000 overlap: 49.00%


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.